# CS221 Mental Health Demo - Kaggle Backend
Notebook này đóng vai trò là Server API (FastAPI) chạy trên GPU của Kaggle để phục vụ cho giao diện Web (Frontend) thông qua ngrok.

In [ ]:
!pip install pyngrok fastapi uvicorn nest-asyncio transformers emoji

In [ ]:
import os
import re
import torch
import emoji
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## 1. Tiền xử lý văn bản (Preprocessing)
Đảm bảo dữ liệu người dùng nhập từ Web được xử lý y hệt như lúc Train.

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Chuyển thành chữ thường
    text = text.lower()
    
    # 2. Thay thế URLs bằng <url>
    text = re.sub(r'http[s]?://\S+', '<url>', text)
    text = re.sub(r'www\.\S+', '<url>', text)
    
    # 3. Thay thế Usernames bằng <username>
    text = re.sub(r'@\w+', '<username>', text)
    
    # 4. Thay thế Numbers bằng <number>
    text = re.sub(r'\b\d+\b', '<number>', text)
    
    # 5. Xử lý Emoji (chuyển thành text mô tả, vd: 😭 -> :loudly_crying_face:)
    text = emoji.demojize(text, delimiters=(" :", ": "))
    
    # 6. Loại bỏ các ký tự đặc biệt (chỉ giữ lại chữ, số và các tag đã tạo)
    text = re.sub(r'[^a-z0-9<>: ]+', ' ', text)
    
    # 7. Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

## 2. Load Model
Điền đường dẫn đến thư mục chứa model của bạn trên Kaggle vào biến `BERT_MODEL_PATH`.

In [ ]:
# Mapping nhãn theo đúng thứ tự lúc train BERT
ID2LABEL = {0: "Normal", 1: "Depression", 2: "Suicidal", 3: "Anxiety"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BERT_MODEL_PATH = "/kaggle/input/models/thaidat733/models-cs221/tensorflow2/default/1/BERT-Finetuning" 

try:
    print("Loading BERT model...")
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_PATH)
    bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL_PATH)
    bert_model.to(device)
    bert_model.eval()
    print("BERT loaded successfully!")
except Exception as e:
    print(f"Could not load BERT model. Error: {e}")
    bert_model = None
    tokenizer = None
    
def predict_with_bert(text):
    if bert_model is None:
        return {"label": "Error (Model not loaded)", "confidence": 0.0}
        
    clean_text = preprocess_text(text)
    
    # Lấy max_length = 300 khớp với lúc Train
    inputs = tokenizer(
        clean_text, 
        truncation=True, 
        padding="max_length", 
        max_length=300, 
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = bert_model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        confidence, predicted_class = torch.max(probs, dim=-1)
        
    label = ID2LABEL[predicted_class.item()]
    conf_score = confidence.item()
    
    return {"label": label, "confidence": conf_score}

## 3. Khởi tạo FastAPI Server

In [ ]:
app = FastAPI()

# Cấu hình CORS để web ở Localhost có thể gọi được API trên Kaggle
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class PredictRequest(BaseModel):
    text: str

@app.get("/")
def home():
    return {"message": "Mental Health API is running!"}

@app.post("/api/predict")
def predict(req: PredictRequest):
    # Gọi hàm dự đoán
    result = predict_with_bert(req.text)
    return result

## 4. Chạy Ngrok và Uvicorn
Điền Authtoken của Ngrok vào để có thể public cổng 8000 của Kaggle ra bên ngoài.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("ngrok")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Đóng các tunnel cũ (nếu có)
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

public_url = ngrok.connect(8000).public_url
print("===============================================================")
print(f"ĐÃ KHỞI TẠO XONG BACKEND API!")
print(f"COPY LINK NÀY VÀO Ô 'API ENDPOINT' TRÊN WEB DEMO CỦA BẠN:\n {public_url}/api/predict")
print("===============================================================")

import asyncio
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()